# Document Extraction with Docling

This notebook demonstrates an end-to-end pipeline for extracting structured information from PDF and image documents using the **Docling** framework. It automatically detects and extracts:

- Tables (as structured CSV files)
- Figures (as cropped PNGs)
- Document text (as clean Markdown)

---

## Set Up Your Environment

Before running the notebook, it's recommended to create a **virtual environment**, Navigate to the project directory where you want to set up the environment:

```bash
python -m venv --system-site-packages docenv
source docenv/bin/activate
pip install ipykernel
python -m ipykernel install --user --name=docenv --display-name "Python (Docling)"

```

---

Ensure that you are in the directory where the Jupyter Notebook and virtual environment is located.
Load the Python (Docling) kernel before running the following cells.


In [ ]:
!. ./docenv/bin/activate; pip install -r requirements.txt

## Import Libraries

This cell imports all required libraries for PDF and image handling, table and figure processing, and data visualization. It includes:
- `os`, `pathlib`, and `glob` for file management,
- `pandas` and `matplotlib` for data display and plotting,
- `PIL.Image` and `fitz` (PyMuPDF) for rendering images from PDF pages,
- Core Docling components for document parsing.

These are necessary to run the extraction pipeline using Docling's PDF processing tools.


In [ ]:
# Set custom directories for cache
from pathlib import Path
import os

def set_env_with_cache_dir(env_var_name: str, subdir: str):
    base_cache = os.path.join(os.getcwd(), ".cache")
    full_path = os.path.join(base_cache, subdir)
    os.environ[env_var_name] = full_path
    os.makedirs(full_path, exist_ok=True)
    print(f"{env_var_name}={full_path}")

set_env_with_cache_dir("PIP_CACHE_DIR", "pip_cache")
set_env_with_cache_dir("HF_HOME", "hf_cache")
set_env_with_cache_dir("EASYOCR_MODULE_PATH", "easyocr_cache")

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import fitz  
import glob
import time 
import hashlib

from docling.datamodel.base_models import InputFormat
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions, TableFormerMode
from docling_core.types.doc.document import PictureItem
from docling.datamodel.accelerator_options import AcceleratorOptions, AcceleratorDevice


## Display the Input File (PDF or Image)

This function displays the first page of a given file.
- For PDFs: Converts the first page to a high-resolution image using `fitz` and renders it with `matplotlib`.
- For images: Directly opens and displays them.
- Unsupported formats raise an error.

This is useful for visually confirming the input document before running the extraction pipeline.


In [ ]:
def display_file(file_path):
    file_extension = os.path.splitext(file_path)[1].lower()
    if file_extension == '.pdf':
        doc = fitz.open(file_path)
        for page_num in range(len(doc)):
            page = doc.load_page(page_num)
            pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))
            image = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            plt.figure(figsize=(16, 12))
            plt.imshow(image)
            plt.axis('off')
            plt.show()
            break
        doc.close()
    elif file_extension in ['.jpg', '.jpeg', '.png', '.bmp', '.tiff']:
        image = Image.open(file_path)
        plt.figure(figsize=(16, 12))
        plt.imshow(image)
        plt.axis('off')
        plt.show()
    else:
        raise ValueError("Unsupported file type.")


### Compute File Hash

This helper function computes a SHA-256 hash of the input PDF or image file.

• Used to detect whether a file has changed since it was last processed  
• Called by `needs_reprocessing()` and inside the extraction function to store metadata


In [ ]:
def compute_file_hash(path):
    hasher = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(8192):
            hasher.update(chunk)
    return hasher.hexdigest()

### Check If File Needs Reprocessing

This function checks whether a file needs to be reprocessed based on:

• A force flag (File will always be reprocessed if this is set to True
• Absence of a previously saved hash file  
• A mismatch between the current and stored file hash

It ensures that already processed documents are not reprocessed, avoiding redundant work.


In [ ]:
def needs_reprocessing(input_path, metadata_path, force=False):
    if force:
        return True
    if not metadata_path.exists():
        return True
    current_hash = compute_file_hash(input_path)
    with open(metadata_path, "r") as f:
        stored_hash = f.read().strip()
    return current_hash != stored_hash

## Extract Tables, Figures, and Markdown using Docling

This function sets up and runs the full Docling document conversion pipeline:
- Configures `PdfPipelineOptions` for layout, table, and figure detection.
- Saves output in three folders:
  - Extracted tables as CSVs 
  - Extracted figures as PNGs
  - Full document markdown
- All outputs are automatically saved to the following directories:
  - `data/output/ExtractedTables`
  - `data/output/ExtractedFigures`
  - `data/output/ExtractedMarkdown`
- These directories are created automatically if they do not already exist.

In [ ]:
def extract_data_with_docling(input_path, output_base="data/output", force=False):
    tables_dir = Path(output_base) / "ExtractedTables"
    figures_dir = Path(output_base) / "ExtractedFigures"
    markdown_dir = Path(output_base) / "ExtractedMarkdown"

    # Create output directories
    tables_dir.mkdir(parents=True, exist_ok=True)
    figures_dir.mkdir(parents=True, exist_ok=True)
    markdown_dir.mkdir(parents=True, exist_ok=True)

    base_name = Path(input_path).stem
    markdown_path = markdown_dir / f"{base_name}.md"
    metadata_path = markdown_dir / f"{base_name}.md.sha256"

    # Check if reprocessing is needed
    reprocess = needs_reprocessing(input_path, metadata_path, force=force)

    if reprocess:
        print(f"Processing {input_path}...")

        # Set up pipeline options
        pipeline_options = PdfPipelineOptions(
            do_table_structure=True,
            do_figure_detection=True,
            images_scale=2.0,
            generate_page_images=True,
            generate_picture_images=True
        )
        pipeline_options.table_structure_options.mode = TableFormerMode.ACCURATE
        pipeline_options.accelerator_options = AcceleratorOptions(device=AcceleratorDevice.CUDA) #Remove this line if you are not using a GPU

        # Convert Input to Docling Document for Processing
        doc_converter = DocumentConverter(
            allowed_formats=[InputFormat.PDF, InputFormat.IMAGE],
            format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)}
        )
        result = doc_converter.convert(input_path)

        # Save markdown
        markdown_text = result.document.export_to_markdown()
        with open(markdown_path, "w", encoding="utf-8") as f:
            f.write(markdown_text)
        print(f"Saved markdown → {markdown_path}")

        # Save file hash
        with open(metadata_path, "w") as f:
            f.write(compute_file_hash(input_path))

        # Save tables
        for table_ix, table in enumerate(result.document.tables):
            df = table.export_to_dataframe()
            table_path = tables_dir / f"{base_name}-table-{table_ix+1}.csv"
            df.to_csv(table_path, index=False)
            print(f"Saved table {table_ix+1} → {table_path}")

        # Extract figures using iterate_items and get_image
        from docling_core.types.doc import PictureItem
        pic_count = 0
        for element, _ in result.document.iterate_items():
            if isinstance(element, PictureItem):
                pic_count += 1
                img = element.get_image(result.document)
                fn = f"{base_name}-figure-{pic_count}.png"
                fig_path = Path(figures_dir)/fn
                img.save(fig_path, "PNG")
                print(f"Saved figure {pic_count} → {fig_path}")

    else:
        print(f"Skipping extraction for {input_path} — outputs are up to date.")

        # Log existing outputs
        if markdown_path.exists():
            print(f"Existing markdown → {markdown_path}")
        existing_tables = sorted(tables_dir.glob(f"{base_name}-table-*.csv"))
        for i, table_file in enumerate(existing_tables, 1):
            print(f"Existing table {i} → {table_file}")
        existing_figures = sorted(figures_dir.glob(f"{base_name}-figure-*.png"))
        for i, fig_file in enumerate(existing_figures, 1):
            print(f"Existing figure {i} → {fig_file}")

    # Always print markdown content
    if markdown_path.exists():
        print("\nMarkdown Output:\n")
        with open(markdown_path, "r", encoding="utf-8") as f:
            print(f.read())

## Display Extracted Tables

This function loads and displays all CSV files corresponding to the input file’s extracted tables.

- It finds all the extracted tables
- Loads each table using `pandas`
- Displays them inline for quick verification

Use this after extraction to verify that tables were parsed and saved correctly.


In [ ]:
def display_extracted_tables(file_path, tables_base="data/output/ExtractedTables"):

    base_stem = Path(file_path).stem
    print("Extracted Tables:\n")
    
    matching_tables = sorted(glob.glob(f"{tables_base}/{base_stem}-table-*.csv"))
    if not matching_tables:
        print("No tables found.")
    for csv_file in matching_tables:
        print(f"Showing: {csv_file}")
        df = pd.read_csv(csv_file)
        display(df)

## Display Extracted Figures

This function visualizes the figures extracted from the input file:
- Searches for saved `.png` images in the output figures directory.
- Displays each one using `matplotlib`.

This is helpful for reviewing whether the figure detection and cropping steps worked as intended.


In [ ]:
def display_extracted_figures(file_path, figures_base="data/output/ExtractedFigures"):

    base_stem = Path(file_path).stem
    print("\nExtracted Figures:\n")
    
    matching_figures = sorted(glob.glob(f"{figures_base}/{base_stem}-figure-*.png"))
    if not matching_figures:
        print("No figures found.")
    for img_path in matching_figures:
        print(f"Showing: {img_path}")
        img = Image.open(img_path)
        plt.figure(figsize=(10, 6))
        plt.imshow(img)
        plt.axis('off')
        plt.show()


## Run Extraction on Your File

This block defines the input file path, displays the document, and prints the markdown output:
- `display_file()` previews the document visually.
- `extract_data_with_docling()` parses it into structured markdown, tables, and figures.

Set the `input_file` variable to the full path of your PDF file (e.g., `"mydocs/report.pdf"`).

Reprocessing can be forced by setting the force flag to True (e.g., `extract_data_with_docling(input_file,force=True)`)

In [ ]:
input_file = "path/to/your/input.pdf"
display_file(input_file)
extract_data_with_docling(input_file)

## View Extracted Tables and Figures

In [ ]:
display_extracted_tables(input_file)

In [ ]:
display_extracted_figures(input_file)